### Inference with Finetuning

In [19]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import matthews_corrcoef, accuracy_score, f1_score,precision_score,recall_score

In [20]:
import sys
from pathlib import Path

parent_dir = str(Path.cwd().parent)
sys.path.append(parent_dir)

#TODO - Add the __init__.py files to avoid this procedure

In [21]:
# Handle tqdm for notebooks
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

from transformers import AutoTokenizer, T5Tokenizer
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer

try:
    from finetuning.src.models import ModelForResidueClassification
    from finetuning.src.dataset_class import ResidueInterfaceDataset
    from finetuning.src.data_utils import load_biodl_dataset
except ImportError:
    print("❌ Error: Could not import 'src'. Make sure you are running this notebook from the root directory containing the 'src' folder.")

In [22]:
def load_tokenizer(model_name):
    print(f"🔹 Loading tokenizer for {model_name}...")
    if any(x in model_name for x in ["ankh", "T5", "t5"]):
        return T5Tokenizer.from_pretrained(model_name, do_lower_case=False, legacy=True)
    elif "esm3" in model_name:
        return EsmSequenceTokenizer()
    else:
        return AutoTokenizer.from_pretrained(model_name)


def load_model(model_name, checkpoint_path, device):
    """
    Loads model weights, supporting both pytorch_model.bin and .safetensors formats.
    """
    print(f"🔹 Loading model architecture: {model_name}")
    model = ModelForResidueClassification(model_name)

    bin_path = os.path.join(checkpoint_path, "pytorch_model.bin")
    safe_path = os.path.join(checkpoint_path, "model.safetensors")

    if os.path.exists(bin_path):
        print(f"🔹 Loading weights from {bin_path}...")
        state_dict = torch.load(bin_path, map_location="cpu")
        model.load_state_dict(state_dict)
    elif os.path.exists(safe_path):
        # safetensors requires its own loader — torch.load won't work here
        print(f"🔹 Loading weights from {safe_path}...")
        from safetensors.torch import load_file
        state_dict = load_file(safe_path, device="cpu")
        model.load_state_dict(state_dict)
    else:
        raise FileNotFoundError(
            f"Could not find 'pytorch_model.bin' or 'model.safetensors' in {checkpoint_path}"
        )

    model.to(device)
    model.eval()
    return model


def prepare_input_data(sequence_str):
    """
    Prepares a single sequence string for the Dataset class.
    Returns seqs, dummy_labels, and a DataFrame (for saving results).
    """
    if not sequence_str or not isinstance(sequence_str, str):
        raise ValueError("INPUT_SEQUENCE must be a non-empty string.")
    seq = sequence_str.strip()
    dummy_labels = [[0] * len(seq)]
    df = pd.DataFrame({'sequence': [seq]})
    return [seq], dummy_labels, df


def run_inference(model, dataloader, seqs, device, max_length=1024):
    """
    Runs inference over a dataloader.
    Returns:
      - all_preds_str: list of binary prediction strings (one per sequence)
      - all_probs:     list of per-residue probability arrays (one per sequence)

    max_length must match the value used in ResidueInterfaceDataset (default 1024).
    Sequences longer than max_length are silently truncated by the dataset, so we
    cap the extraction window here to avoid indexing beyond valid logits.
    The usable residue window is [1 : max_length-1] (positions 0 and max_length-1
    are reserved for BOS and EOS tokens respectively).
    """
    all_preds_str = []
    all_probs = []
    max_residues = max_length - 2  # exclude BOS (pos 0) and EOS (pos max_length-1)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            # Note: attention_mask is unused by ESM3 inside the model but
            # is required by the non-ESM3 branch — always safe to pass.

            outputs = model(input_ids, attention_mask)
            logits = outputs["logits"]

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int().cpu().numpy()
            probs_np = probs.cpu().numpy()

            batch_start = len(all_preds_str)
            for i in range(len(preds)):
                global_idx = batch_start + i
                original_len = len(seqs[global_idx])

                # Cap to the actual number of residues the dataset encoded.
                # Sequences longer than max_length-2 were truncated at dataset time,
                # so we must not try to read more positions than were encoded.
                effective_len = min(original_len, max_residues)

                # Skip BOS token (index 0), extract exactly effective_len residues
                start_idx = 1
                end_idx = start_idx + effective_len

                valid_preds = preds[i][start_idx:end_idx]
                valid_probs = probs_np[i][start_idx:end_idx]

                if original_len > max_residues:
                    print(f"⚠️  Sequence {global_idx} (len={original_len}) was truncated to {max_residues} residues.")

                all_preds_str.append("".join(map(str, valid_preds)))
                all_probs.append(valid_probs)

    return all_preds_str, all_probs

---
## Configuration

In [23]:
# Model identifier (must match what was used during training)
MODEL_NAME = "EvolutionaryScale/esm3-sm-open-v1"

# Path to the checkpoint folder containing pytorch_model.bin or model.safetensors

CHECKPOINT_PATH = "saved_models/best_biolip" #checkpoint pdbBind-vdw
#CHECKPOINT_PATH = "saved_models/best_biolip"

BATCH_SIZE = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

sys.path.append(os.getcwd())

---
## Part 1 — Single Sequence Inference

In [24]:
# Input: replace with your protein sequence
INPUT_SEQUENCE = "MKTVRQERLKSIVRILEAAKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"

# Output CSV path
SINGLE_OUTPUT_FILE = "single_prediction.csv"

In [25]:
print(f"🚀 Starting single-sequence inference on {DEVICE}")

# Prepare data
print(f"🔹 Preparing input sequence (Length: {len(INPUT_SEQUENCE)})...")
seqs, dummy_labels, result_df = prepare_input_data(INPUT_SEQUENCE)

# Load resources
tokenizer = load_tokenizer(MODEL_NAME)
model = load_model(MODEL_NAME, CHECKPOINT_PATH, DEVICE)

# Build dataset & dataloader
dataset = ResidueInterfaceDataset(seqs, dummy_labels, tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# Run inference
print("🔹 Running inference...")
preds_str, probs = run_inference(model, dataloader, seqs, DEVICE)

# Save results
result_df["prediction"] = preds_str
result_df["probabilities"] = [p.tolist() for p in probs]
result_df.to_csv(SINGLE_OUTPUT_FILE, index=False)

print(f"\n✅ Inference complete! Results saved to {SINGLE_OUTPUT_FILE}")
print("\nResult:")
print(f"Sequence:   {result_df.iloc[0]['sequence']}")
print(f"Prediction: {result_df.iloc[0]['prediction']}")

🚀 Starting single-sequence inference on cuda
🔹 Preparing input sequence (Length: 65)...
🔹 Loading tokenizer for EvolutionaryScale/esm3-sm-open-v1...
🔹 Loading model architecture: EvolutionaryScale/esm3-sm-open-v1
🔹 Loading weights from saved_models/best_biolip/pytorch_model.bin...
🔹 Running inference...


Processing:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Inference complete! Results saved to single_prediction.csv

Result:
Sequence:   MKTVRQERLKSIVRILEAAKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG
Prediction: 00001100010001000001110000000110100110011001101111110111110010110


---
## Part 2 — Test Set Evaluation (Accuracy, MCC, F1)

In [26]:
# Path to the test CSV (same format expected by load_biodl_dataset)
TEST_CSV_PATH = "data/final_zk448_test.csv"

# Dataset type (must match what was used during training, e.g. 'p')
DATASET_TYPE = "p"

# Output CSV path for per-sequence results
TEST_OUTPUT_FILE = "test_set_predictions.csv"

In [27]:
print(f"🚀 Starting test set evaluation on {DEVICE}")

# Make sure your imports include the new metrics:
# from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, precision_score, recall_score

# -----------------------------------------------
# A. Load test data (sequences + ground truth labels)
# -----------------------------------------------
print(f"🔹 Loading test set from {TEST_CSV_PATH}...")
test_seqs, test_labels = load_biodl_dataset(TEST_CSV_PATH, dataset_type=DATASET_TYPE)
print(f"   Found {len(test_seqs)} sequences.")

# -----------------------------------------------
# B. Build dataset & dataloader
# -----------------------------------------------
test_dataset = ResidueInterfaceDataset(test_seqs, test_labels, tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------------------------
# C. Run inference
# -----------------------------------------------
print("🔹 Running inference on test set...")
test_preds_str, test_probs = run_inference(model, test_dataloader, test_seqs, DEVICE)

# -----------------------------------------------
# D. Compute metrics (flattened over all residues)
# -----------------------------------------------
MAX_RESIDUES = 1024 - 2  # must match max_length used in dataset (1024) minus BOS and EOS

all_true = []
all_pred = []

for i, (pred_str, true_labels) in enumerate(zip(test_preds_str, test_labels)):
    pred_arr = np.array(list(pred_str), dtype=int)
    # Truncate true labels to match what the dataset actually encoded
    true_arr = np.array(true_labels[:MAX_RESIDUES], dtype=int)

    # Sanity check: lengths must match after truncation
    if len(pred_arr) != len(true_arr):
        print(f"⚠️  Length mismatch at sequence {i}: pred={len(pred_arr)}, true={len(true_arr)}. Skipping.")
        continue

    all_true.extend(true_arr.tolist())
    all_pred.extend(pred_arr.tolist())

all_true = np.array(all_true)
all_pred = np.array(all_pred)

accuracy  = accuracy_score(all_true, all_pred)
mcc       = matthews_corrcoef(all_true, all_pred)
precision = precision_score(all_true, all_pred, zero_division=0) # Added Precision
recall    = recall_score(all_true, all_pred, zero_division=0)    # Added Recall
f1        = f1_score(all_true, all_pred, zero_division=0)
f1_macro  = f1_score(all_true, all_pred, average="macro", zero_division=0)

print("\n" + "="*45)
print("          TEST SET METRICS (residue-level)")
print("="*45)
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  MCC       : {mcc:.4f}")
print(f"  Precision : {precision:.4f}") # Added Output
print(f"  Recall    : {recall:.4f}")    # Added Output
print(f"  F1 (pos)  : {f1:.4f}")
print(f"  F1 (macro): {f1_macro:.4f}")
print(f"  Total residues evaluated: {len(all_true)}")
print("="*45)

# -----------------------------------------------
# E. Save per-sequence results to CSV
# -----------------------------------------------
per_seq_rows = []
for i, (seq, pred_str, true_labels, probs_arr) in enumerate(
    zip(test_seqs, test_preds_str, test_labels, test_probs)
):
    pred_arr = np.array(list(pred_str), dtype=int)
    true_arr = np.array(true_labels[:MAX_RESIDUES], dtype=int)  # match truncation

    if len(pred_arr) != len(true_arr):
        continue  # already warned above

    seq_acc       = accuracy_score(true_arr, pred_arr)
    seq_mcc       = matthews_corrcoef(true_arr, pred_arr) if len(np.unique(true_arr)) > 1 else float("nan")
    seq_precision = precision_score(true_arr, pred_arr, zero_division=0) # Added seq Precision
    seq_recall    = recall_score(true_arr, pred_arr, zero_division=0)    # Added seq Recall
    seq_f1        = f1_score(true_arr, pred_arr, zero_division=0)

    per_seq_rows.append({
        "sequence":       seq[:MAX_RESIDUES],  # show only the evaluated portion
        "true_labels":    "".join(map(str, true_arr)),
        "prediction":     pred_str,
        "seq_accuracy":   round(seq_acc, 4),
        "seq_mcc":        round(seq_mcc, 4) if not np.isnan(seq_mcc) else "N/A",
        "seq_precision":  round(seq_precision, 4), # Added to dict
        "seq_recall":     round(seq_recall, 4),    # Added to dict
        "seq_f1":         round(seq_f1, 4),
    })

results_df = pd.DataFrame(per_seq_rows)
results_df.to_csv(TEST_OUTPUT_FILE, index=False)
print(f"\n✅ Per-sequence results saved to {TEST_OUTPUT_FILE}")

🚀 Starting test set evaluation on cuda
🔹 Loading test set from data/final_zk448_test.csv...
   Found 336 sequences.
🔹 Running inference on test set...


/orfeo/LTS/LADE/LT_storage/bio_data/ppi/PPI-Reps/finetuning/src/data_utils.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: str(x).replace(",", "").strip())


Processing:   0%|          | 0/336 [00:00<?, ?it/s]


          TEST SET METRICS (residue-level)
  Accuracy  : 0.8291
  MCC       : 0.5388
  Precision : 0.5278
  Recall    : 0.7760
  F1 (pos)  : 0.6282
  F1 (macro): 0.7586
  Total residues evaluated: 84941

✅ Per-sequence results saved to test_set_predictions.csv
